<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 9 </h1></center>
<center><h2> Évaluer un RAG </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 9 · 20 à 25 min · ≈ 0,01 $ · prépare le Lab 5 et le chapitre 7 du projet**
> Matériau : le jeu de 30 questions du projet et des exécutions **déjà enregistrées**. Seul le correcteur LLM appelle le modèle (quelques dizaines d'appels). **Sans gateway, la démo se rabat sur les chiffres enregistrés.**

```
 1. SANS MODÈLE   la preuve est-elle dans ce que le modèle a reçu ?   gratuit, stable
 2. AVEC MODÈLE   un correcteur LLM note les réponses                  à auditer
 3. LE BRUIT      deux exécutions identiques ne donnent pas le même    à mesurer
                  résultat
```

In [1]:
import os
import warnings
from collections import Counter

# Move to the repository root when Jupyter starts the kernel in the demo folder.
if os.path.basename(os.getcwd()) == "demo":
    os.chdir("../..")

from dotenv import load_dotenv

load_dotenv()
os.environ["LANGSMITH_TRACING"] = "false"  # no trace sent during the demo
warnings.filterwarnings("ignore")  # no library warning on the projector

from src import evaluation

questions = evaluation.load_reference()
by_id = {question["id"]: question for question in questions}
runs = {name: evaluation.load_run(name) for name in ("baseline_1", "baseline_2", "titled_1", "titled_2", "agent_v2_1", "agent_v2_2")}
print(len(questions), "questions |", len(runs), "exécutions enregistrées :", ", ".join(runs))

30 questions | 6 exécutions enregistrées : baseline_1, baseline_2, titled_1, titled_2, agent_v2_1, agent_v2_2


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Le jeu de questions </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **30 questions** dont on connaît la bonne réponse : un jeu de référence.
- Chaque ligne : la question, **les fiches attendues**, la réponse attendue, **la phrase de preuve**.
- **Hors corpus** : 8 questions où la bonne réponse est de refuser.

In [2]:
print(dict(Counter(question["type"] for question in questions)))
print()

item = by_id["q03"]
print("Question  :", item["question"])
print("Fiche     :", item["gold_doc_ids"])
print("Attendu   :", item["reference_answer"])
print("Preuve    :", item["evidence"])

{'simple': 12, 'multi': 10, 'unanswerable': 8}

Question  : Mon fils de 8 ans a la grippe et j'ai deux enfants à charge. Combien de jours par an puis-je m'absenter pour le garder ?
Fiche     : ['F151']
Attendu   : 3 jours par an au maximum (la convention collective peut accorder plus). Le plafond est de 5 jours si l'enfant a moins d'1 an ou si le salarié a au moins 3 enfants à charge de moins de 16 ans.
Preuve    : La durée maximale du congé pour enfant malade est fixée à 3 jours par an.


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Mesure 1 : a-t-on trouvé les bonnes fiches ? </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- On compare **sans modèle** : les fiches attendues sont-elles dans celles envoyées ?
- Sur les **22 questions qui ont une réponse**, pour `baseline` et `titled`.

In [3]:
answerable = [question for question in questions if question["type"] != "unanswerable"]

missed = {}
for version in ("baseline_1", "titled_1"):
    result_of = {result["id"]: result for result in runs[version]}
    missed[version] = [q["id"] for q in answerable if not evaluation.fiches_found(q, result_of[q["id"]])]
    found = len(answerable) - len(missed[version])
    print(f"{version:11} fiches trouvées : {found}/22 | ratées : {', '.join(missed[version])}")

print()
if len(missed["baseline_1"]) == len(missed["titled_1"]):
    print("❌ Même score, et pourtant deux assistants différents")

baseline_1  fiches trouvées : 18/22 | ratées : q03, q12, q14, q27
titled_1    fiches trouvées : 18/22 | ratées : q11, q13, q14, q27

❌ Même score, et pourtant deux assistants différents


- **18 contre 18** : la mesure ne voit **aucune différence**.
- Pourtant les deux versions **ne ratent pas les mêmes questions**.
- **Une mesure trop grossière donne le même score à deux assistants différents.**

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Mesure 2 : la preuve est-elle dans le contexte ? </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Plus fine : les **phrases de preuve** sont-elles dans le **texte exact** envoyé au modèle ?
- On relance sur les **deux exécutions** de chaque version : le résultat bouge-t-il ?

In [4]:
def count_evidence(version):
    """How many answerable questions have all their evidence in the text sent to the model."""
    result_of = {result["id"]: result for result in runs[version]}
    return sum(evaluation.evidence_in_context(q, result_of[q["id"]]) for q in answerable)


def count_wrong_refusals(version):
    """How many answerable questions the assistant refused."""
    result_of = {result["id"]: result for result in runs[version]}
    return sum(evaluation.is_refusal(result_of[q["id"]]) for q in answerable)


print(f"{'exécution':12} {'preuve dans le contexte':>24} {'refus à tort':>14}")
for version in ("baseline_1", "baseline_2", "titled_1", "titled_2"):
    print(f"{version:12} {count_evidence(version):>21}/22 {count_wrong_refusals(version):>12}/22")

print()
print("q03 : preuve dans le contexte ?")
for version in ("baseline_1", "titled_1"):
    result = next(result for result in runs[version] if result["id"] == "q03")
    found = evaluation.evidence_in_context(by_id["q03"], result)
    print("  ", version, "✅ oui" if found else "❌ non")

exécution     preuve dans le contexte   refus à tort
baseline_1                      11/22            3/22
baseline_2                      11/22            3/22
titled_1                        17/22            1/22
titled_2                        17/22            1/22

q03 : preuve dans le contexte ?
   baseline_1 ❌ non
   titled_1 ✅ oui


- **11 → 15** : avec le titre et la section entière, la preuve arrive **plus souvent** devant le modèle.
- **Identique dans les deux exécutions** : le gain est **réel**. Refus à tort : **3 → 1**, stable aussi.
- **Sans modèle, gratuit, reproductible** : la première mesure à faire.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Le correcteur LLM, audité </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Pour savoir si la réponse est **juste**, il faut la lire. À grande échelle : **un modèle correcteur**.
- Mais **qui corrige le correcteur ?** On le compare à **notre propre note**, posée avant de regarder.
- Ci-dessous, **12 réponses de `titled`** : note humaine contre correcteur (verdict enregistré).

In [5]:
HUMAN = {"q01": "correct", "q03": "faux", "q04": "partiel", "q05": "correct", "q09": "correct", "q10": "correct",
         "q12": "correct", "q14": "faux", "q16": "faux", "q26": "correct", "q28": "correct", "q30": "partiel"}

titled_result = {result["id"]: result for result in runs["titled_1"]}

agreements = 0
print(f"{'question':9} {'humain':9} {'correcteur':11}")
for question_id, human_note in HUMAN.items():
    grader_note = titled_result[question_id]["verdict"]
    agree = human_note == grader_note
    agreements += agree
    print(f"{question_id:9} {human_note:9} {grader_note:11} {'✅' if agree else '❌ désaccord'}")
print(f"\nAccords : {agreements}/12")

question  humain    correcteur 
q01       correct   correct     ✅
q03       faux      faux        ✅
q04       partiel   partiel     ✅
q05       correct   partiel     ❌ désaccord
q09       correct   correct     ✅
q10       correct   correct     ✅
q12       correct   partiel     ❌ désaccord
q14       faux      faux        ✅
q16       faux      partiel     ❌ désaccord
q26       correct   partiel     ❌ désaccord
q28       correct   partiel     ❌ désaccord
q30       partiel   partiel     ✅

Accords : 7/12


- **7 accords sur 12** après correction du prompt (**4 sur 12** avec le premier : trop sévère).
- Presque tous les désaccords : le correcteur répond **« partiel »** à une réponse que l'humain juge **correcte**.
- Refaisons la mesure **en direct**, avec le premier prompt puis le second.

In [6]:
from typing import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


class FirstGrade(BaseModel):
    """First version of the grader: a reason, then a verdict."""

    reason: str = Field(description="Une phrase qui compare la réponse à la réponse attendue")
    verdict: Literal["correct", "partiel", "faux"] = Field(
        description="correct : tous les faits de la réponse attendue sont là ; partiel : il en manque ; faux : un fait est faux ou contredit")


FIRST_PROMPT = """Vous corrigez les réponses d'un assistant sur les congés des salariés.
On vous donne la question, la réponse attendue (juste) et la réponse de l'assistant.

- correct : tous les faits de la réponse attendue figurent dans la réponse, sans erreur.
- partiel : la réponse est juste mais il manque un fait de la réponse attendue.
- faux : la réponse contient un fait faux ou contredit la réponse attendue."""


def grade_first(item, answer):
    """Verdict of the first (strict) grader."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
    text = ("Question : " + item["question"] + "\n\nRéponse attendue : " + item["reference_answer"]
            + "\n\nRéponse de l'assistant : " + answer)
    return llm.with_structured_output(FirstGrade).invoke([SystemMessage(FIRST_PROMPT), HumanMessage(text)]).verdict


def grade_second(item, answer):
    """Verdict of the final grader of the project (it lists the asked facts first)."""
    return evaluation.grade_answer(item, answer).verdict


def live_grades(grade_function):
    """Grade the 12 answers with one grader. Returns None if the gateway is unreachable."""
    try:
        return {qid: grade_function(by_id[qid], titled_result[qid]["answer"]) for qid in HUMAN}
    except Exception as error:
        print("⚠️ gateway inaccessible :", type(error).__name__)
        return None


first = live_grades(grade_first)
second = live_grades(grade_second)
second_again = live_grades(grade_second)

if first and second and second_again:
    print(f"{'question':9} {'humain':9} {'prompt 1':9} {'prompt 2':9} {'prompt 2 (bis)':9}")
    for qid, human_note in HUMAN.items():
        cells = [first[qid], second[qid], second_again[qid]]
        marks = ["✅" if cell == human_note else "❌" for cell in cells]
        print(f"{qid:9} {human_note:9} {first[qid]:9} {marks[0]} {second[qid]:9} {marks[1]} {second_again[qid]:9} {marks[2]}")
    for label, grades in (("prompt 1", first), ("prompt 2", second), ("prompt 2 (bis)", second_again)):
        print(f"{label:15} accords : {sum(grades[qid] == HUMAN[qid] for qid in HUMAN)}/12")
    print("\nValeurs de référence (chapitre 7) : 4/12 puis 7/12 ; en direct, vos valeurs varient.")
else:
    print("Sortie enregistrée (chapitre 7) : 4/12 accords avec le premier prompt, 7/12 avec le second ; en direct, vos valeurs varient.")

question  humain    prompt 1  prompt 2  prompt 2 (bis)
q01       correct   correct   ✅ correct   ✅ correct   ✅
q03       faux      partiel   ❌ faux      ✅ faux      ✅
q04       partiel   partiel   ✅ partiel   ✅ partiel   ✅
q05       correct   partiel   ❌ partiel   ❌ partiel   ❌
q09       correct   partiel   ❌ correct   ✅ correct   ✅
q10       correct   partiel   ❌ correct   ✅ partiel   ❌
q12       correct   partiel   ❌ partiel   ❌ partiel   ❌
q14       faux      faux      ✅ faux      ✅ faux      ✅
q16       faux      partiel   ❌ partiel   ❌ partiel   ❌
q26       correct   partiel   ❌ partiel   ❌ partiel   ❌
q28       correct   partiel   ❌ partiel   ❌ partiel   ❌
q30       partiel   partiel   ✅ partiel   ✅ partiel   ✅
prompt 1        accords : 4/12
prompt 2        accords : 7/12
prompt 2 (bis)  accords : 6/12

Valeurs de référence (chapitre 7) : 4/12 puis 7/12 ; en direct, vos valeurs varient.


- **Le premier prompt** s'accorde peu : il voit **« partiel »** partout (**4/12**, chapitre 7).
- **Le second** fait mieux (**7/12** au chapitre 7), mais **relancé, il ne donne pas toujours les mêmes notes** : **en direct, vos valeurs varient**.
- **Un correcteur LLM se mesure comme un instrument** : comparer, lire les désaccords, corriger, remesurer.
- **Il sert à trier** (« à relire en premier »), **pas à départager** deux versions à une question près.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. Le bruit </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- À température 0, **la même question ne donne pas toujours la même réponse**.
- On compare **deux exécutions de la même version** : textes et verdicts qui changent.

In [7]:
pairs = [("baseline", "baseline_1", "baseline_2"), ("titled", "titled_1", "titled_2"), ("agent", "agent_v2_1", "agent_v2_2")]

print(f"{'version':10} {'texte différent':>16} {'verdict différent':>18}")
for label, first_name, second_name in pairs:
    first_run, second_run = runs[first_name], runs[second_name]
    text_changed = sum(a["answer"] != b["answer"] for a, b in zip(first_run, second_run))
    verdict_changed = sum(a["verdict"] != b["verdict"] for a, b in zip(first_run, second_run))
    print(f"{label:10} {text_changed:>13}/30 {verdict_changed:>15}/30")

version     texte différent  verdict différent
baseline               5/30               4/30
titled                 6/30               1/30
agent                 16/30               8/30


- Le texte change pour **plusieurs questions sur 30**, le verdict pour **quelques-unes**. L'agent bouge le plus.
- **Règle : un écart n'existe que s'il dépasse l'écart entre deux exécutions de la même version.**
- Appliquons-la aux mesures de la démo.

In [8]:
summaries = {name: evaluation.summarize(questions, runs[name]) for name in ("baseline_1", "baseline_2", "titled_1", "titled_2")}

measures = [("fiches trouvées", "fiches_found"), ("preuve dans le contexte", "evidence_in_context"),
            ("refus à tort", "wrongly_refused"), ("réponses correctes", "correct"), ("réponses fausses", "wrong")]

print(f"{'mesure':26} {'baseline':>10} {'titled':>10}   verdict")
for label, key in measures:
    base = [summaries["baseline_1"][key], summaries["baseline_2"][key]]
    titled = [summaries["titled_1"][key], summaries["titled_2"][key]]
    if max(base) < min(titled) or max(titled) < min(base):
        verdict = "✅ écart réel"
    elif base == titled:
        verdict = "➖ aucun écart"
    else:
        verdict = "❌ dans le bruit"
    print(f"{label:26} {str(base):>10} {str(titled):>10}   {verdict}")

mesure                       baseline     titled   verdict
fiches trouvées              [18, 18]   [18, 18]   ➖ aucun écart
preuve dans le contexte      [11, 11]   [17, 17]   ✅ écart réel
refus à tort                   [3, 3]     [1, 1]   ✅ écart réel
réponses correctes             [8, 7]     [8, 8]   ❌ dans le bruit
réponses fausses               [0, 3]     [2, 1]   ❌ dans le bruit


![](assets/09_comparison.png)

- **Preuve dans le contexte** et **refus à tort** : écarts **réels**, stables.
- **Réponses correctes / fausses** : **dans le bruit**. On **ne dit pas** « `titled` donne de meilleures réponses ».
- **On garde `titled` pour ce que la mesure montre** : plus de preuves devant le modèle, moins de refus à tort.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 6. Une source valide n'est pas une réponse juste </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- `q03` (enfant malade) : la fiche est trouvée, **la preuve est dans le contexte**…
- … et la réponse est quand même fausse. **Chercher juste est nécessaire, pas suffisant.**

In [9]:
def mark(ok):
    return "✅" if ok else "❌"


item = by_id["q03"]
result = titled_result["q03"]

context = result["context"]
position = context.find("3 jours par an")
print("Contexte envoyé :", len(context), "caractères")
print("…" + context[max(0, position - 120):position + 40].replace("\n", " ") + "…")
print()
print("Attendu  :", item["reference_answer"][:40], "…")
print("Réponse  :", result["answer"])
print()
print(mark(evaluation.fiches_found(item, result)), "Fiche attendue trouvée")
print(mark(evaluation.evidence_in_context(item, result)), "Preuve dans le contexte")
print(mark(result["verdict"] == "correct"), "Réponse :", result["verdict"].upper(), "(5 jours au lieu de 3)")

Contexte envoyé : 5230 caractères
…a durée du congé du salarié pour enfant malade ? > Autre cas] La durée maximale du congé pour enfant malade est fixée à 3 jours par an.   La convention collecti…

Attendu  : 3 jours par an au maximum (la convention …
Réponse  : Vous pouvez vous absenter jusqu'à 5 jours par an pour garder votre fils malade, car vous avez deux enfants à charge.

✅ Fiche attendue trouvée
✅ Preuve dans le contexte
❌ Réponse : FAUX (5 jours au lieu de 3)


In [10]:
to_reread = [(q["id"], titled_result[q["id"]]["verdict"]) for q in answerable
             if evaluation.evidence_in_context(q, titled_result[q["id"]]) and titled_result[q["id"]]["verdict"] != "correct"]

print("Preuve dans le contexte, mais réponse pas « correct » :", len(to_reread), "questions")
print(", ".join(question_id + " (" + verdict + ")" for question_id, verdict in to_reread))

Preuve dans le contexte, mais réponse pas « correct » : 10 questions
q03 (faux), q04 (partiel), q05 (partiel), q12 (partiel), q16 (partiel), q25 (partiel), q26 (partiel), q28 (partiel), q29 (partiel), q30 (partiel)


- Le modèle avait « 3 jours par an » **et** le cas particulier des 5 jours : il a pris **le mauvais cas**.
- Cette liste = **les réponses à relire en premier**, pas un score (le correcteur s'y trompe parfois : q05, q12, q26 et q28 sont « correct » pour l'humain).
- **Retrouver ce genre de cas en production : c'est la trace** → Part 12.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > Récapitulatif </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| Ce que la démo a montré | À retenir | Où vous le mettez en pratique |
|---|---|---|
| **Fiches trouvées** : 18 / 18 | mesure trop grossière, elle ne voit rien | **Lab 5** |
| **Preuve dans le contexte** : 11 → 15 | sans modèle, stable : la première mesure | **Lab 5**, **chapitre 7** |
| **Correcteur LLM** : 4 puis 7 accords sur 12 | à auditer contre une note humaine, sert à trier | **Lab 5**, **chapitre 7** |
| **Bruit** : deux exécutions, deux résultats | un écart sous le bruit ne prouve rien | **Lab 5**, **chapitre 7** |
| **Source valide ≠ réponse juste** (`q03`) | la preuve est nécessaire, pas suffisante | **Chapitre 7** (section 7.8) |

- **Retenez l'ordre** : mesures **sans modèle** → correcteur **audité** → écart comparé au **bruit**.
- **Prochain pas : le Lab 5**, puis la Part 10 (agents).